In [ ]:
# Colab 請選 2026.07；安裝到獨立環境，避免更改核心已載入的 NumPy。
from pathlib import Path
import subprocess
import sys
import tempfile

assert sys.version_info[:3] == (3, 12, 13), "請使用書籍或 Colab 2026.07 的 Python 3.12.13"
python = sys.executable
if "google.colab" in sys.modules:
    %pip -q install virtualenv
    env = Path("/content/cheminfo-models-env")
    if not (env / "bin/python").is_file():
        subprocess.run([python, "-m", "virtualenv", "-q", str(env)], check=True)
    python = str(env / "bin/python")
    pins = ["deepchem==2.8.1.dev20260821122258", "numpy==1.26.4", "scipy==1.17.1",
            "rdkit==2026.9.1", "scikit-learn==1.9.1", "pandas==3.0.6", "matplotlib==3.11.2"]
    subprocess.run([python, "-m", "pip", "install", "-q", *pins], check=True)
    subprocess.run([python, "-m", "pip", "check"], check=True)
work = tempfile.TemporaryDirectory()
script = Path(work.name) / "09-11.py"

## 模型怎麼選：資料量與計算成本

首格準備執行環境，下一格寫出分析程式，末格執行並顯圖。練習可接在分析程式末端。第一次執行需連線取得 MoleculeNet 資料。

In [ ]:
%%writefile "{script}"
from pathlib import Path
import os
import tempfile
import deepchem as dc
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

cache = Path(os.environ.get("DEEPCHEM_DATA_DIR", Path(tempfile.gettempdir()) / "book-moleculenet"))
cache.mkdir(parents=True, exist_ok=True)
options = dict(featurizer=dc.feat.MACCSKeysFingerprint(), splitter="scaffold",
               transformers=[], data_dir=str(cache), save_dir=str(cache), reload=False)
folder = Path("../../figures") if Path("../../_quarto.yml").is_file() else Path("figures")
folder.mkdir(exist_ok=True)

from time import perf_counter
from threadpoolctl import threadpool_limits
from sklearn.base import clone
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import RBF
from sklearn.linear_model import Ridge
from sklearn.svm import SVR
from sklearn.metrics import mean_absolute_error

_, (train, valid, test), _ = dc.molnet.load_delaney(**options)
X = np.asarray(train.X, dtype=float)
y = train.y[:, 0]
order = np.random.default_rng(42).permutation(len(y))
sizes = [100, 300, 800]
models = {"Ridge": Ridge(alpha=1.0, solver="cholesky"),
          "SVR": SVR(C=1.0, gamma="scale"),
          "Exact GP": GaussianProcessRegressor(kernel=RBF(4.0), alpha=0.1,
                                               normalize_y=True, optimizer=None)}
results = {}
with threadpool_limits(limits=1):
    for name, prototype in models.items():
        clone(prototype).fit(X[order[:40]], y[order[:40]])  # warm-up outside timing
        for n in sizes:
            idx = order[:n]
            Xn, yn = X[idx], y[idx]
            times = []
            for repeat in range(3):
                model = clone(prototype)
                start = perf_counter()
                model.fit(Xn, yn)
                times.append(perf_counter() - start)
            err = mean_absolute_error(valid.y[:, 0], model.predict(valid.X))
            results[name, n] = np.median(times), err
            print(f"{name:9s} n={n}: median fit {np.median(times):.5f}s; validation MAE {err:.4f}")
fig, axes = plt.subplots(1, 2, figsize=(9, 4), layout="constrained")
for name in models:
    vals = np.array([results[name, n] for n in sizes])
    for j, ax in enumerate(axes):
        ax.plot(sizes, vals[:, j], marker="o", label=name)
axes[0].set(yscale="log", ylabel="Median fit time (s, log axis)")
axes[1].set_ylabel("Validation MAE (logS)")
for ax in axes:
    ax.set_xlabel("Training records")
    ax.set_xticks(sizes)
axes[0].legend(frameon=False)
fig.savefig(folder / "09-11-cost-quality.png", dpi=160)

In [ ]:
from IPython.display import Image, display
subprocess.run([python, str(script)], check=True)
out = Path("../../figures") if Path("../../_quarto.yml").is_file() else Path("figures")
display(Image(filename=str(out / "09-11-cost-quality.png")))